In [1]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


# BFloat16 Adder Module

This notebook incrementally shows how to build a floating point addition circuit for the bfloat16 format.  

The design is inspired by [this paper](https://ieeexplore.ieee.org/abstract/document/10440496). 

## Floating point addition algorithm

### Stage 1: Input Parsing and Extraction
- **Objective**: Extract the basic components (sign, exponent, mantissa) from the input numbers.
- **Steps**:
  1. Extract the sign bits from both input numbers.
  2. Extract the exponent bits from both input numbers.
  3. Extract the mantissa bits, including the hidden implicit bit.

### Stage 2: Exponent Difference Calculation
- **Objective**: Align the exponents of the two numbers.
- **Steps**:
  1. Subtract the exponent of the second input from the first to find the offset.
  2. Determine which number has the larger exponent.
  3. Align the mantissa of the number with the smaller exponent by right shifting.

### Stage 3: Mantissa Alignment and Bit Preparation
- **Objective**: Prepare mantissas for addition.
- **Steps**:
  1. Apply the necessary right shifts to the smaller mantissa for alignment.
  2. Generate Sticky, Guard, and Round (SGR) bits to facilitate rounding.

### Stage 4: Mantissa Addition/Subtraction
- **Objective**: Add or subtract the aligned mantissas.
- **Steps**:
  1. Perform addition or subtraction based on the sign bits.
  2. Handle negative results by inverting if necessary.
  3. Use a Leading Zero Detector (LZD) to find leading zeros in the result.

### Stage 5: Normalization and Final Adjustments
- **Objective**: Finalize the result by normalization and rounding.
- **Steps**:
  1. Normalize the result using the LZD output to adjust the exponent.
  2. Implement rounding and correct for any overflow by adjusting the exponent.
  3. Determine the final sign of the result.

### Block Diagram of 5 stage FP32 Adder

<img src="./assets/float-adder.png" width="500">

# PyRTL Implementation

### Imports & Setup

In [828]:
from IPython.display import display_svg
from typing import List, Tuple
import random
import torch
import numpy as np
import pandas as pd
from itertools import combinations_with_replacement
from typing import Callable, Any
import pyrtl
from pyrtl.rtllib.libutils import twos_comp_repr, rev_twos_comp_repr
from pyrtl import (
    WireVector, 
    Const, 
    Input,
    Output, 
    Register, 
    Simulation, 
    SimulationTrace, 
    reset_working_block
)
from kai.src.utils import custom_render_trace, basic_circuit_analysis
from kai.src.bfloat16 import BF16

In [735]:
pyrtl.set_debug_mode(False)
# pyrtl.set_debug_mode(True)

In [736]:
E_BITS  = 8
M_BITS  = 7
MSB     = E_BITS + M_BITS

### Repr Funcs

In [799]:
def repr_bf16(x):
    binary = format(x, "016b")
    bf16_value = BF16(x, True)
    return f"{binary[:1]}.{binary[1:9]}.{binary[9:]} ({bf16_value.tensor:.3f})"

def repr_bf16_tensor(x):
    return f"{torch.tensor(x, dtype=torch.uint16).view(torch.bfloat16):.6f}"

def repr_sign(x):
    return "0 (+)" if x == 0 else "1 (-)"

def repr_exp(x):
    return f"{format(x, '08b')} (unbiased={x - 127})"

def repr_mantissa(x):
    binary = format(x, "08b")
    decimal = int(binary[0], 2) + int(binary[1:], 2) / (2**7)
    return f"{binary[:1]}.{binary[1:]} ({decimal:.3f})"

def repr_mantissa_hidden(x):
    binary = format(x, "07b")
    decimal = 1 + int(binary, 2) / (2**7)
    return f"1.{binary} ({decimal:.3f})"

def repr_ext_mantissa(x):
    binary = format(x, "016b")
    decimal = int(binary[0], 2) + int(binary[1:], 2) / (2**15)
    return f"{binary[:1]}.{binary[1:]} ({decimal:.3f})"

def repr_mantissa_sum(x):
    binary = format(x, "09b")
    decimal = int(binary[:2], 2) + int(binary[2:], 2) / (2**7)
    return f"{binary[:2]}.{binary[2:]} ({decimal:.3f})"

def repr_num(x):
    return format(x, '0b') + f" ({x})"

## Stage 1: Data Extraction

In [19]:
def extract_sign(input_a: WireVector, input_b: WireVector, msb: int) -> Tuple[WireVector, WireVector]:
    sign_a = WireVector(1, name='sign_a')
    sign_b = WireVector(1, name='sign_b')
    
    sign_a <<= input_a[msb]  # Assuming the MSB is the sign bit
    sign_b <<= input_b[msb]
    
    return sign_a, sign_b

In [20]:
def extract_exponent(input_a: WireVector, input_b: WireVector, e_bits: int) -> Tuple[WireVector, WireVector]:
    exp_a = WireVector(e_bits, name='exp_a')
    exp_b = WireVector(e_bits, name='exp_b')
    
    exp_a <<= input_a[-(1+e_bits):-1]
    exp_b <<= input_b[-(1+e_bits):-1]
    
    return exp_a, exp_b

In [21]:
def extract_mantissa(input_a: WireVector, input_b: WireVector, m_bits: int) -> Tuple[WireVector, WireVector]:
    mantissa_a = WireVector(m_bits+1, name='mantissa_a')
    mantissa_b = WireVector(m_bits+1, name='mantissa_b')
    
    # Concatenate the implicit leading 1
    mantissa_a <<= pyrtl.concat(Const(1, 1), input_a[:m_bits])
    mantissa_b <<= pyrtl.concat(Const(1, 1), input_b[:m_bits])
    
    return mantissa_a, mantissa_b

In [22]:
def stage_1(input_a: WireVector, input_b: WireVector, e_bits: int, m_bits: int):
    # Extract components
    sign_a, sign_b = extract_sign(input_a, input_b, e_bits + m_bits)
    exp_a, exp_b = extract_exponent(input_a, input_b, e_bits)
    mantissa_a, mantissa_b = extract_mantissa(input_a, input_b, m_bits)
    return sign_a, sign_b, exp_a, exp_b, mantissa_a, mantissa_b

In [82]:
def test_stage1(e_bits=E_BITS, m_bits=M_BITS):
    reset_working_block()
    total_bits = 1 + e_bits + m_bits
    # Inputs should match the bfloat16 format: 1 sign bit, 8 exponent bits, and 7 mantissa bits
    input_a = Input(total_bits, name='input_a')
    input_b = Input(total_bits, name='input_b')

    # Extract components
    stage_1(input_a, input_b, e_bits, m_bits)
    
    # Simulate and test
    sim_trace = SimulationTrace()
    sim = Simulation(tracer=sim_trace)
    sim.step({'input_a': 0b0100000001000000, 'input_b': 0b0100000010000000})  # Example inputs

    input_repr_map = {
        'input_a': repr_bf16,
        'input_b': repr_bf16,
        'sign_a': repr_sign,
        'sign_b': repr_sign,
        'exp_a': repr_exp,
        'exp_b': repr_exp,
        'mantissa_a': repr_mantissa,
        'mantissa_b': repr_mantissa,
    }
    trace_list = list(input_repr_map.keys())

    custom_render_trace(sim_trace, trace_list=trace_list, repr_per_name=input_repr_map)

test_stage1()


<IPython.core.display.Javascript object>

## Stage 2: Exponent Subtraction and Detection

In [124]:
def calculate_exponent_difference(exp_a: WireVector, exp_b: WireVector):
    assert len(exp_a) == len(exp_b)
    exp_diff = WireVector(len(exp_a)+1, name='exp_diff')
    exp_greater = WireVector(len(exp_a), name='exp_greater')

    exp_diff <<= pyrtl.signed_sub(exp_a, exp_b)  # This can be negative, indicating which is larger
    exp_greater <<= pyrtl.mux(exp_diff[8], exp_a, exp_b)
    return exp_diff, exp_greater

In [743]:
def stage_2(
    sign_a: WireVector,
    sign_b: WireVector,
    exp_a: WireVector, 
    exp_b: WireVector, 
    mant_a: WireVector,
    mant_b: WireVector,
    e_bits: int, 
    m_bits: int
):
    assert len(sign_a) == len(sign_b) == 1
    assert len(exp_a)  == len(exp_b)  == e_bits
    assert len(mant_a) == len(mant_b) == m_bits + 1

    sign_xor     = WireVector(1, name='sign_xor')
    exp_diff     = WireVector(e_bits+1, name='exp_diff')
    signed_shift = WireVector(e_bits+1, name='signed_shift')
    exp_larger   = WireVector(e_bits, name='exp_larger')
    mant_smaller = WireVector(m_bits+1, name='mant_smaller')
    mant_larger  = WireVector(m_bits+1, name='mant_larger')

    sign_xor <<= sign_a ^ sign_b

    exp_diff     <<= exp_a - exp_b  # This can be negative, indicating which is larger
    exp_larger   <<= pyrtl.mux(exp_diff[e_bits], exp_a, exp_b)
    signed_shift <<= pyrtl.mux(
        exp_diff[e_bits], 
        exp_diff[:e_bits], 
        pyrtl.concat(exp_diff[e_bits], (~exp_diff[:e_bits] + 1)[:e_bits])
    )

    # Select the smaller mantissa to align and the larger to keep unchanged
    mant_smaller <<= pyrtl.mux(exp_diff[e_bits], mant_b, mant_a)
    mant_larger  <<= pyrtl.mux(exp_diff[e_bits], mant_a, mant_b)

    return sign_xor, exp_larger, signed_shift, mant_smaller, mant_larger

In [713]:
def test_stage_2():
    reset_working_block()

    sign_a, sign_b  = Input(1, 'sign_a')        , Input(1, 'sign_b')
    exp_a, exp_b    = Input(E_BITS, 'exp_a')    , Input(E_BITS, 'exp_b')
    mant_a, mant_b  = Input(M_BITS+1, 'mant_a') , Input(M_BITS+1, 'mant_b')

    sxr, exp_larger, shift_amount, mant_smaller, mant_larger = stage_2(sign_a, sign_b, exp_a, exp_b, mant_a, mant_b, E_BITS, M_BITS)

    sxr_out = Output(1, 'sxr_out')
    exp_larger_out = Output(E_BITS, 'exp_larger_out')
    shift_amount_out = Output(E_BITS+1, 'shift_amount_out')
    mant_smaller_out = Output(M_BITS+1, 'mant_smaller_out')
    mant_larger_out = Output(M_BITS+1, 'mant_larger_out')

    sxr_out <<= sxr
    exp_larger_out <<= exp_larger
    shift_amount_out <<= shift_amount
    mant_smaller_out <<= mant_smaller
    mant_larger_out <<= mant_larger
    
    sim_trace = pyrtl.SimulationTrace()
    sim = pyrtl.Simulation(tracer=sim_trace)

    inputs = {
        'sign_a': [0,0,0,0],
        'sign_b': [0,0,0,0],
        'exp_a': [1, 128, 129, 255],
        'exp_b': [128, 255, 128, 200],
        'mant_a': [129, 133, 200, 192],
        'mant_b': [160, 192, 160, 148]
    }
    sim.step_multiple(inputs)  # Example inputs

    # Representation maps for visualization
    def repr_signed_shift(x):
        binary = format(x, f'0{E_BITS+1}b')
        sign = '(+)' if binary[0]=='0' else '(-)'
        return f"{sign} {binary[1:]} ({int(binary[1:], 2)})"
    
    input_repr_map = {
        'exp_a': repr_exp,
        'exp_b': repr_exp,
        'mant_a': repr_mantissa,
        'mant_b': repr_mantissa,
        'exp_larger': repr_exp,
        'exp_diff': lambda x: rev_twos_comp_repr(x, E_BITS+1),
        'signed_shift': repr_signed_shift,
        'mant_smaller': repr_mantissa,
        'mant_larger': repr_mantissa,
    }
    trace_list = list(input_repr_map.keys())

    custom_render_trace(sim_trace, trace_list=trace_list, repr_per_name=input_repr_map, repr_func=repr_num)

test_stage_2()

<IPython.core.display.Javascript object>

#### Extras

In [85]:
def align_mantissa(mantissa_a, mantissa_b, exp_diff):
    aligned_mantissa_a = pyrtl.WireVector(len(mantissa_a), name='aligned_mantissa_a')
    aligned_mantissa_b = pyrtl.WireVector(len(mantissa_b), name='aligned_mantissa_b')
    
    # Shift mantissas based on exponent difference
    shift_amount = pyrtl.abs(exp_diff)  # Ensure non-negative shift amount
    
    aligned_mantissa_a <<= pyrtl.mux(exp_diff >= 0, mantissa_a, mantissa_a >> shift_amount)
    aligned_mantissa_b <<= pyrtl.mux(exp_diff < 0, mantissa_b, mantissa_b >> shift_amount)
    
    return aligned_mantissa_a, aligned_mantissa_b

In [86]:
def stage_2(exp_a, exp_b, mantissa_a, mantissa_b):
    exp_diff, larger_exp = calculate_exponent_difference(exp_a, exp_b)
    aligned_mantissa_a, aligned_mantissa_b = align_mantissa(mantissa_a, mantissa_b, exp_diff)
    return exp_diff, larger_exp, aligned_mantissa_a, aligned_mantissa_b


In [87]:
def test_stage2(e_bits=E_BITS, m_bits=M_BITS):
    pyrtl.reset_working_block()
    total_bits = 1 + e_bits + m_bits
    input_a = pyrtl.Input(total_bits, name='input_a')
    input_b = pyrtl.Input(total_bits, name='input_b')

    # Stage 1 outputs
    sign_a, sign_b, exp_a, exp_b, mantissa_a, mantissa_b = stage_1(input_a, input_b, e_bits, m_bits)

    # Stage 2 processing
    exp_diff, larger_exp, aligned_mantissa_a, aligned_mantissa_b = stage_2(exp_a, exp_b, mantissa_a, mantissa_b)

    # Outputs for testing
    exp_diff_out = pyrtl.Output(name='exp_diff')
    larger_exp_out = pyrtl.Output(name='larger_exp')
    aligned_mantissa_a_out = pyrtl.Output(name='aligned_mantissa_a')
    aligned_mantissa_b_out = pyrtl.Output(name='aligned_mantissa_b')

    exp_diff_out <<= exp_diff
    larger_exp_out <<= larger_exp
    aligned_mantissa_a_out <<= aligned_mantissa_a
    aligned_mantissa_b_out <<= aligned_mantissa_b

    # Simulate and test
    sim_trace = pyrtl.SimulationTrace()
    sim = pyrtl.Simulation(tracer=sim_trace)
    sim.step({'input_a': 0b0100000001000000, 'input_b': 0b0100000010000000})  # Example inputs

    # Representation maps for visualization
    input_repr_map = {
        'input_a': repr_bf16,
        'input_b': repr_bf16,
        'sign_a': repr_sign,
        'sign_b': repr_sign,
        'exp_a': repr_exp,
        'exp_b': repr_exp,
        'mantissa_a': repr_mantissa,
        'mantissa_b': repr_mantissa,
        'exp_diff': str,
        'larger_exp': repr_exp,
        'aligned_mantissa_a': repr_mantissa,
        'aligned_mantissa_b': repr_mantissa,
    }
    trace_list = list(input_repr_map.keys())

    custom_render_trace(sim_trace, trace_list=trace_list, repr_per_name=input_repr_map)

test_stage2()


AttributeError: module 'pyrtl' has no attribute 'abs'

## Stage 3: Mantissa Alignment and SGR Generator

### Align Mantissa

In [310]:
def align_mantissa(
    mant_smaller: WireVector,
    shift_amount: WireVector,
    m_bits: int
) -> WireVector:
    """
    Align the smaller mantissa by shifting right
    
    Args:
        mant_smaller: mantissa to be shifted (m_bits + 1 wide)
        shift_amount: number of positions to shift right (e_bits wide)
        m_bits: number of mantissa bits (7 for bfloat16)
    
    Returns:
        aligned_mantissa_msb, aligned_mantissa_lsb: shifted mantissa
    """
    assert len(mant_smaller) == m_bits + 1
    assert len(shift_amount) == E_BITS
    
    # Detect if shift amount exceeds mantissa width
    max_useful_shift = 2 * (m_bits + 1)
    
    # Clamp shift amount to maximum useful value
    clamped_shift = WireVector(E_BITS, 'clamped_shift')
    clamped_shift <<= pyrtl.select(
        shift_amount > max_useful_shift,
        max_useful_shift,
        shift_amount,
    )
    
    # Perform the right shift
    extended_mantissa = pyrtl.concat(mant_smaller, pyrtl.Const(0, m_bits+1))

    aligned_mantissa = WireVector(max_useful_shift, name='aligned_mantissa')
    assert len(extended_mantissa) == len(aligned_mantissa)

    aligned_mantissa <<= pyrtl.shift_right_logical(extended_mantissa, clamped_shift)
    return pyrtl.chop(aligned_mantissa, *[m_bits+1]*2)

def test_align_mantissa():
    reset_working_block()
    
    mant_smaller = Input(M_BITS + 1, 'mant_smaller')
    shift_amount = Input(E_BITS, 'shift_amount')
    
    aligned_msb, aligned_lsb = align_mantissa(mant_smaller, shift_amount, M_BITS)
    aligned_msb_out = Output(M_BITS+1, 'aligned_msb_out')
    aligned_lsb_out = Output(M_BITS+1, 'aligned_lsb_out')
    aligned_msb_out <<= aligned_msb
    aligned_lsb_out <<= aligned_lsb

    sim_trace = SimulationTrace()
    sim = Simulation(tracer=sim_trace)
    
    inputs = {
        'mant_smaller': [0b10000000, 0b10100000, 0b11000000, 0b10000000],
        'shift_amount': [1, 9, 3, 45]
    }
    
    sim.step_multiple(inputs)
    input_repr_map = {
        'mant_smaller': repr_mantissa,
        'shift_amount': repr_num,
        'clamped_shift': str,
        'aligned_mantissa': repr_ext_mantissa,
        'aligned_msb_out': repr_mantissa,
        'aligned_lsb_out': lambda x: format(x, f'0{M_BITS+1}b'),
    }
    
    trace_list = list(input_repr_map.keys())
    custom_render_trace(sim_trace, trace_list=trace_list, repr_per_name=input_repr_map)

print("Testing align_mantissa:")
test_align_mantissa()

Testing align_mantissa:


<IPython.core.display.Javascript object>

### SGR Generator

In [304]:
def generate_sgr(
    aligned_mant_lsb: WireVector,
    m_bits: int
) -> tuple[WireVector, WireVector, WireVector]:
    """
    Generate sticky, guard, and round bits
    
    Args:
        mant_smaller: original mantissa before shifting (m_bits + 1 wide)
        shift_amount: number of positions shifted right (e_bits wide)
        m_bits: number of mantissa bits (7 for bfloat16)
    
    Returns:
        sticky_bit, guard_bit, round_bit
    """
    assert len(aligned_mant_lsb) == m_bits + 1
    
    guard_bit = WireVector(1, name='guard_bit')
    round_bit = WireVector(1, name='round_bit')
    sticky_bit = WireVector(1, name='sticky_bit')
    
    guard_bit <<= aligned_mant_lsb[m_bits]
    round_bit <<= aligned_mant_lsb[m_bits-1]
    sticky_bit <<= pyrtl.or_all_bits(aligned_mant_lsb[:m_bits-1])
            
    return sticky_bit, guard_bit, round_bit

def test_generate_sgr():
    reset_working_block()
    
    aligned_mant_lsb = Input(M_BITS + 1, 'aligned_mant_lsb')
    
    sticky_bit, guard_bit, round_bit = generate_sgr(aligned_mant_lsb, M_BITS)
    
    sticky_out = Output(1, 'sticky_out')
    guard_out = Output(1, 'guard_out')
    round_out = Output(1, 'round_out')
    
    sticky_out <<= sticky_bit
    guard_out <<= guard_bit
    round_out <<= round_bit
    
    sim_trace = SimulationTrace()
    sim = Simulation(tracer=sim_trace)
    
    inputs = {
        'aligned_mant_lsb': [0b10000000, 0b01000000, 0b10100000, 0b11000000, 0b00100000],
    }
    
    sim.step_multiple(inputs)
    
    input_repr_map = {
        'aligned_mant_lsb': lambda x: format(x, f'0{M_BITS+1}b'),
        'sticky_out': repr_num,
        'guard_out': repr_num,
        'round_out': repr_num
    }
    
    trace_list = list(input_repr_map.keys())
    custom_render_trace(sim_trace, trace_list=trace_list, repr_per_name=input_repr_map)

print("\nTesting generate_sgr:")
test_generate_sgr()



Testing generate_sgr:


<IPython.core.display.Javascript object>

### Putting it all together

In [731]:
def stage_3(
    mant_smaller: WireVector,
    shift_amount: WireVector,
    m_bits: int
):
    """
    Combine alignment and SGR generation
    """
    aligned_mant_msb, aligned_mant_lsb = align_mantissa(mant_smaller, shift_amount, m_bits)
    sticky_bit, guard_bit, round_bit = generate_sgr(aligned_mant_lsb, m_bits)
    
    return aligned_mant_msb, sticky_bit, guard_bit, round_bit


## Stage 4: Mantissa Add & Leading Zero Detection

### Mantissa Addition/Subtraction

In [714]:
def add_sub_mantissas(
    mant_aligned: WireVector,
    mant_unchanged: WireVector,
    sign_xor: WireVector,
    m_bits: int
) -> tuple[WireVector, WireVector]:
    """
    Perform addition or subtraction on mantissas based on signs
    
    Args:
        mant_aligned: aligned mantissa (m_bits + 1 wide)
        mant_unchanged: unchanged mantissa (m_bits + 1 wide)
        sign_xor: result of XOR on the input signs
        m_bits: number of mantissa bits
        
    Returns:
        result_mantissa: result of add/sub (m_bits + 2 wide to handle overflow)
        is_negative: whether the result is negative
    """
    assert len(mant_aligned) == len(mant_unchanged) == m_bits + 1
    assert len(sign_xor) == 1

    raw_result = WireVector(m_bits + 3, 'raw_result')
    with pyrtl.conditional_assignment:
        with sign_xor:
            raw_result |= mant_unchanged.zero_extended(m_bits+2) - mant_aligned.zero_extended(m_bits+2)
        with pyrtl.otherwise:
            raw_result |= mant_unchanged.zero_extended(m_bits+2) + mant_aligned.zero_extended(m_bits+2)

    # Detect if result is negative
    is_negative = WireVector(1, 'is_negative')
    is_negative <<= raw_result[m_bits+2]  # MSB indicates sign
    
    # If result is negative, convert back to positive
    abs_result = WireVector(m_bits + 2, 'abs_result')
    abs_result <<= pyrtl.select(is_negative, ~raw_result + 1, raw_result)
    
    return abs_result, is_negative


In [723]:
def test_add_sub():
    reset_working_block()
    
    # Create inputs
    mant_aligned = Input(M_BITS + 1, 'mant_aligned')
    mant_unchanged = Input(M_BITS + 1, 'mant_unchanged')
    sign_xor = Input(1, 'sign_xor')
    
    # Get outputs
    result, is_neg = add_sub_mantissas(mant_aligned, mant_unchanged, sign_xor, M_BITS)
    
    # Create output wires
    result_out = Output(M_BITS + 2, 'result_out')
    is_neg_out = Output(1, 'is_neg_out')
    
    result_out <<= result
    is_neg_out <<= is_neg
    
    # Simulate
    sim_trace = SimulationTrace()
    sim = Simulation(tracer=sim_trace)
    
    # Test cases:
    # 1. Simple addition (same signs)
    # 2. Simple subtraction (different signs)
    # 3. Result becomes negative
    # 4. Large numbers that might overflow
    inputs = {
        'mant_aligned':   [0b10000000, 0b10000000, 0b11000000, 0b11111111],
        'mant_unchanged': [0b10000000, 0b11000000, 0b10000000, 0b11111111],
        'sign_xor':        [0,          1,          1,          0],
    }
    
    sim.step_multiple(inputs)
    
    input_repr_map = {
        'mant_aligned': repr_mantissa,
        'mant_unchanged': repr_mantissa,
        # 'effective_operation': lambda x: 'Subtract' if x else 'Add',
        'raw_result': lambda x: format(x, f'0{M_BITS + 3}b'),
        'result_out': repr_mantissa_sum,
        'is_neg_out': repr_sign
    }
    
    trace_list = list(input_repr_map.keys())
    custom_render_trace(sim_trace, trace_list=trace_list, repr_per_name=input_repr_map)

test_add_sub()


<IPython.core.display.Javascript object>

### Leading Zero Detector

#### Algorithm

This algorithm efficiently counts leading zeros in a binary number using a tree-based structure. It works by encoding pairs of bits and then merging these encodings upward through a tree until we get a final count.

**1. Pair Encoding Stage**

First, we encode pairs of bits according to their leading zero count:

| Input Pair | Encoding | Meaning |
|------------|----------|---------|
| 00         | 10       | 2 leading zeros |
| 01         | 01       | 1 leading zero |
| 10         | 00       | 0 leading zeros |
| 11         | 00       | 0 leading zeros |


**2. Merging Stage**

After encoding pairs, we merge adjacent encodings using these rules:

1. If both sides start with 1:
   - Output "10...0" (pad with zeros to match width)
2. If left side starts with 1 and right starts with 0:
   - Output "01" + right[1:] (concatenate "01" with rest of right side)
3. If left side starts with 0:
   - Keep left section (preserves leading zero count)
   
##### Why does this work?

1. **Pair Encoding**: The initial encoding captures local leading zero information
2. **Merging Rules**: Preserve leading zero count while combining sections
3. **Tree Structure**: Allows O(log n) computation depth
4. **Final Count**: Number of 1s in final encoding equals leading zero count

#### Example Walkthrough


Let's trace through an 8-bit example: `00000111`

1. Split: 
   - `00` | `00` | `01` | `11`
2. Encode: 
   - `10` | `10` | `01` | `00`
   - 00 → 10 (2 zeros)
   - 00 → 10 (2 zeros)
   - 01 → 01 (1 zero)
   - 11 → 00 (0 zeros)
3. Merge:
   - `100` | `001`
   - (10,10) → 100 (both start with 1, rule 1)
   - (01,00) → 001 (starts with 0, rule 3)
4. Merge:
   - `0101`
   - $0101_2 = 5_{10}$
   
The final encoding 0101 contains 5 ones, indicating 5 leading zeros in the original number.



#### Pure Python Implementation

In [469]:
def encode_pair(pair: str) -> str:
    """Encode a pair of bits according to leading zeros"""
    if pair == "00": return "10"   # 2 leading zeros
    if pair == "01": return "01"   # 1 leading zero
    return "00"                    # 0 leading zeros (10 or 11)

def merge_encoded(left: str, right: str) -> str:
    """Merge two encoded sections"""
    if len(left) == 0: return right
    if len(right) == 0: return left
    
    # If left starts with 0, keep left section
    if left[0] == '1' and right[0] == '1':
        return '1' + '0' * len(left)
    elif left[0] == '1':
        return '01' + right[1:]
    if left[0] == '0':
        return '0'+left

def lzd_algorithm(binary: str, verbose=False) -> int:
    """Count leading zeros using the tree-based algorithm"""
    if not binary: return 0
    
    # Step 1: Split input into pairs
    split_input = [binary[i:i+2] for i in range(0, len(binary), 2)]
    
    # Step 2: Encode pairs
    encoded = [encode_pair(pair) for pair in split_input]
    
    if verbose:
        print(f"Input: {binary}")
        print(f"Split input: {split_input}")
        print(f"After pair encoding: {encoded}")
    
    # Step 3 & 4: Merge encodings until we have one result
    while len(encoded) > 1:
        new_encoded = []
        for i in range(0, len(encoded)-1, 2):
            new_encoded.append(merge_encoded(encoded[i], encoded[i+1]))
        if len(encoded) % 2:
            new_encoded.append(encoded[-1])
        encoded = new_encoded
        if verbose:
            print(f"After merge: {encoded}")
    
    algorithm_result = int(encoded[0], 2)
    if verbose:
        print(f"Result: {algorithm_result}")
    return algorithm_result

def lzd_9bit(binary: str):
    """Count leading zeros of 9 bit numbers"""
    assert len(binary) == 9, "Input must be 9 bits long"
    # First check the msb
    if binary[0] == '1':
        return 0
    # Then check the next 8 bits
    return lzd_algorithm(binary[1:])+1

def lzd_count(binary: str) -> int:
    """Ground truth leading zero counter"""
    true_result = 0
    for bit in binary:
        if bit == '0':
            true_result += 1
        else:
            break
    return true_result

# Test cases
def test_lzd_fn(n, bits=9):
    """Generate n test cases and evaluate results"""
    cases = random.sample(range(2**bits), n)
    binary_cases = list(map(lambda x: format(x, f'0{bits}b'), cases))

    # Test the 9-bit algorithm
    alg_results = [lzd_9bit(case) for case in binary_cases]
    true_results = [lzd_count(case) for case in binary_cases]

    df = pd.DataFrame(
        {
            'binary': binary_cases,
            'alg_result': alg_results, 
            'true_result': true_results
        }
    )
    failed = df[df['alg_result'] != df['true_result']]
    if failed.empty:
        print(f"All {n} test cases passed!")
    else:
        print(f"{len(failed)} test cases failed:")
        print(failed)

test_lzd_fn(500)

# Examples
example_cases = [
    "00000111",  # 8 bits
    "0000",      # 4 bits
    "1000",      # 4 bits
]

for case in example_cases:
    lzd_algorithm(case, verbose=True)
    print("-" * 50)

All 500 test cases passed!
Input: 00000111
Split input: ['00', '00', '01', '11']
After pair encoding: ['10', '10', '01', '00']
After merge: ['100', '001']
After merge: ['0101']
Result: 5
--------------------------------------------------
Input: 0000
Split input: ['00', '00']
After pair encoding: ['10', '10']
After merge: ['100']
Result: 4
--------------------------------------------------
Input: 1000
Split input: ['10', '00']
After pair encoding: ['00', '10']
After merge: ['000']
Result: 0
--------------------------------------------------


#### Now lets build the hardware with PyRTL

First we build a LZD for an 8-bit input

In [511]:
def enc2(d: WireVector) -> WireVector:
    """
    Encode 2 bits into their leading zero count representation
    
    Args:
        d: 2-bit input WireVector
        
    Returns:
        2-bit WireVector encoding the leading zeros:
        00 -> 10 (2 leading zeros)
        01 -> 01 (1 leading zero)
        10 -> 00 (0 leading zeros)
        11 -> 00 (0 leading zeros)
    """
    result = WireVector(2)
    with pyrtl.conditional_assignment:
        with d == 0b00:
            result |= 0b10
        with d == 0b01:
            result |= 0b01
        with pyrtl.otherwise:
            result |= 0b00
    return result

def clzi(left: WireVector, right: WireVector, n: int) -> WireVector:
    """
    Merge two encoded vectors in the leading zero count tree
    
    Args:
        left: Left section to process
        right: Right section to process
        n: Size of each pair
        
    Returns:
        Merged and encoded output vector
    """
    assert len(left) == len(right) == n
    result = WireVector(n+1)
    
    left_msb, right_msb = left[-1], right[-1]
    
    with pyrtl.conditional_assignment:
        with left_msb & right_msb:                  # both sides start with 1
            result |= Const(1<<n, bitwidth=n+1)     # n leading zeros

        with left_msb:                              # left starts with 1
            result |= pyrtl.concat(
                Const(0b01, bitwidth=2),            # 01
                right[:n-1]                         # right[:MSB-1]
            )

        with pyrtl.otherwise:                       # left starts with 0
            result |= pyrtl.concat(
                Const(0, bitwidth=1),               # 0
                left                                # Rest from left section
            )
    
    return result

def leading_zero_count_8bit(value: WireVector, m_bits: int) -> WireVector:
    """
    Calculate leading zeros for a 8-bit input (for bf16 mantissa addition)
    
    Args:
        value: 8-bit input WireVector (mantissa with hidden bit and overflow bit)
        
    Returns:
        4-bit WireVector containing the count of leading zeros (max 8 zeros)
    """
    assert len(value) == m_bits+1, "Input must be 8 bits wide"

    # First level: encode pairs of bits (4 pairs total for 8 bits)
    # Results in a list with 4 2-bit WireVectors, indexed from MSB [0] to LSB [-1]
    pairs = pyrtl.chop(value, *[2 for _ in range((m_bits+1)//2)])
    encoded_pairs = [enc2(pair) for pair in pairs]

    first_merge = [
        clzi(encoded_pairs[0], encoded_pairs[1], 2),  # First group
        clzi(encoded_pairs[2], encoded_pairs[3], 2)   # Last group (handles remaining bits)
    ]

    final_result = WireVector(4, 'lzc_result')
    final_result <<= clzi(first_merge[0], first_merge[1], 3)
    return final_result

def test_leading_zero_count_8bit():
    """Test the 8-bit leading zero counter implementation"""
    pyrtl.reset_working_block()
    
    input_val = pyrtl.Input(8, 'input_val')
    lzc_out = pyrtl.Output(4, 'lzc_out')
    
    lzc_out <<= leading_zero_count_8bit(input_val, M_BITS)
    
    sim_trace = pyrtl.SimulationTrace()
    sim = pyrtl.Simulation(tracer=sim_trace)
    
    # Test vectors for 8-bit values
    test_values = [
        0b10000000,  # 0 leading zeros
        0b01000000,  # 1 leading zero
        0b00100000,  # 2 leading zeros
        0b00000001,  # 7 leading zeros
        0b00000000   # 8 leading zeros
    ]
    
    sim.step_multiple({'input_val': test_values})
    
    # Custom representation for better readability
    input_repr_map = {
        'input_val': lambda x: format(x, '08b'),
        'lzc_out': str,
    }
    
    trace_list = list(input_repr_map.keys())
    custom_render_trace(sim_trace, trace_list=trace_list, repr_per_name=input_repr_map)

test_leading_zero_count_8bit()

<IPython.core.display.Javascript object>

Now we can build the overall module for the 9-bit result from the previous step

In [519]:
def leading_zero_detector_module(mantissa_sum: WireVector, m_bits: int) -> WireVector:
    """
    Calculate leading zeros in mantissa sum, accounting for overflow bit
    
    Args:
        mantissa_sum: sum of mantissas (m_bits + 2 wide to handle overflow)
        m_bits: number of mantissa bits in original format
        
    Returns:
        lzc: leading zero count (4 bits wide to count up to 9 zeros)
        
    The module checks the overflow (carry) bit to determine if normalization is needed:
    - If carry_bit is 1: no leading zeros (return 0)
    - If carry_bit is 0: count leading zeros in remaining bits and add 1
    """
    assert len(mantissa_sum) == m_bits+2, f"Input must be {m_bits+2} bits wide"

    # Use the carry bit from the sum result to determine if we use the LZC or not
    carry_bit = mantissa_sum[-1]

    lzc = WireVector(4, 'leading_zero_count')
    lzc <<= pyrtl.select(carry_bit, 0, leading_zero_count_8bit(mantissa_sum[:-1], m_bits)+1)
    return lzc

def test_leading_zero_detector_module():
    """Test the 9-bit leading zero detector module"""
    reset_working_block()

    mant_sum_in = Input(9, 'mant_sum_in')
    leading_zero_detector_module(mant_sum_in, M_BITS)
    
    sim_trace = pyrtl.SimulationTrace()
    sim = pyrtl.Simulation(tracer=sim_trace)
    
    # Test vectors for 9-bit values
    test_values = [
        0b100000000,  # 0 leading zeros
        0b010000000,  # 1 leading zero
        0b110000000,
        0b001000000,  # 2 leading zeros
        0b000001101,
        0b000000001,  # 8 leading zeros
        0b000000000   # 9 leading zeros
    ]
    sim.step_multiple({'mant_sum_in': test_values})

    input_repr_map = {
        'mant_sum_in': repr_mantissa_sum,
        'leading_zero_count': repr_num,
    }
    trace_list = list(input_repr_map.keys())
    custom_render_trace(sim_trace, trace_list=trace_list, repr_per_name=input_repr_map)

test_leading_zero_detector_module()

leading_zero_count/4W:
Wire Traceback, most recent call last 
  File "<frozen runpy>", line 198, in _run_module_as_main
   File "<frozen runpy>", line 88, in _run_code
   File "/opt/homebrew/Caskroom/miniconda/base/envs/dsc180/lib/python3.12/site-packages/ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
   File "/opt/homebrew/Caskroom/miniconda/base/envs/dsc180/lib/python3.12/site-packages/traitlets/config/application.py", line 1075, in launch_instance
    app.start()
   File "/opt/homebrew/Caskroom/miniconda/base/envs/dsc180/lib/python3.12/site-packages/ipykernel/kernelapp.py", line 739, in start
    self.io_loop.start()
   File "/opt/homebrew/Caskroom/miniconda/base/envs/dsc180/lib/python3.12/site-packages/tornado/platform/asyncio.py", line 205, in start
    self.asyncio_loop.run_forever()
   File "/opt/homebrew/Caskroom/miniconda/base/envs/dsc180/lib/python3.12/asyncio/base_events.py", line 641, in run_forever
    self._run_once()
   File "/opt/homebrew/Cas

<IPython.core.display.Javascript object>

### Putting it all together

In [725]:
def stage_4(
    mant_aligned: WireVector,
    mant_unchanged: WireVector,
    sign_xor: WireVector,
    m_bits: int
):
    """
    Add/sub the mantissas based on input signs and count the leading zeros in the result
    
    Args:
        mant_aligned: aligned mantissa (m_bits + 1 wide)
        mant_unchanged: unchanged mantissa (m_bits + 1 wide)
        sign_xor: XOR of input signs
        m_bits: number of mantissa bits
        
    Returns:
        result_mantissa: result of add/sub (m_bits + 2 wide to handle overflow)
        is_negative: whether the result is negative
        lzc: leading zero count (4 bits wide to count up to 9 zeros)
    """
    
    mantissa_sum, is_neg = add_sub_mantissas(mant_aligned, mant_unchanged, sign_xor, m_bits)
    lzc = leading_zero_detector_module(mantissa_sum, m_bits)
    return mantissa_sum, is_neg, lzc

In [728]:
def test_stage_4():
    reset_working_block()
    
    # Create inputs
    mant_aligned = Input(M_BITS + 1, 'mant_aligned')
    mant_unchanged = Input(M_BITS + 1, 'mant_unchanged')
    sign_xor = Input(1, 'sign_xor')
    
    # Get outputs
    mantissa_sum, is_neg, lzc = stage_4(mant_aligned, mant_unchanged, sign_xor, M_BITS)
    
    # Create output wires
    mant_sum_out = Output(M_BITS + 2, 'mant_sum_out')
    is_neg_out = Output(1, 'is_neg_out')
    lzc_out = Output(4, 'lzc_out')
    
    mant_sum_out <<= mantissa_sum
    is_neg_out <<= is_neg
    lzc_out <<= lzc
    
    # Simulate
    sim_trace = SimulationTrace()
    sim = Simulation(tracer=sim_trace)
    
    # Test cases:
    # 1. Simple addition (same signs)
    # 2. Simple subtraction (different signs)
    # 3. Result becomes negative
    # 4. Large numbers that might overflow
    inputs = {
        'mant_aligned':   [0b10000000, 0b10000000, 0b11000000, 0b11111111, 0b01000000],
        'mant_unchanged': [0b10000000, 0b11000000, 0b10000000, 0b11111111, 0b10000000],
        'sign_xor':        [0,          1,          1,          0,           0],
    }
    
    sim.step_multiple(inputs)
    
    input_repr_map = {
        'mant_aligned': repr_mantissa,
        'mant_unchanged': repr_mantissa,
        'mant_sum_out': repr_mantissa_sum,
        'is_neg_out': repr_sign,
        'lzc_out': repr_num,
    }
    
    trace_list = list(input_repr_map.keys())
    custom_render_trace(sim_trace, trace_list=trace_list, repr_per_name=input_repr_map)

test_stage_4()


<IPython.core.display.Javascript object>

## Stage 5: Sign Detection, Normalization and Rounding

In this step, the LZD offset obtained in the previous
step is subtracted from the exponent of the larger number.

Similarly, the final mantissa is obtained after the normalization
and rounding steps. 

Also, In the LZD, it is assumed that the
point is to the right after the carry bit (the point is shifted one
bit to the left). 
Therefore, the final exponent needs to be
increased by 1. 

Moreover, after rounding, an overflow may
occur; in this case, the exponent must be updated again by
incrementing it by one. Also in this stage, the sign detection
logic chooses the final sign.

### Sign Detection Logic

In [719]:
def detect_final_sign(
    sign_a: WireVector,
    sign_b: WireVector,
    exp_diff: WireVector,
    is_neg: WireVector,
    e_bits: int,
) -> WireVector:
    """
    Determine the final sign of the floating point addition result
    
    Args:
        sign_a: sign bit of first number
        sign_b: sign bit of second number
        exp_diff: difference between exponents (exp_a - exp_b)
        is_neg: sign from mantissa addition/subtraction
        e_bits: number of exponent bits
        
    Returns:
        final_sign: computed sign bit for the result
    """
    assert len(sign_a) == len(sign_b) == len(is_neg) == 1
    assert len(exp_diff) == E_BITS + 1  # Signed difference
    
    final_sign = WireVector(1, 'final_sign')
    
    # Check if signs are the same
    same_signs = ~(sign_a ^ sign_b)
    
    # For same signs case, use either input sign (they're the same)
    # For different signs case, use magnitude comparison
    with pyrtl.conditional_assignment:
        with same_signs:
            final_sign |= sign_a
        with pyrtl.otherwise:
            # If exponents are equal (exp_diff == 0), use mantissa comparison
            # Otherwise, use sign of number with larger exponent
            with exp_diff[:e_bits-1] == 0:
                final_sign |= is_neg
            with exp_diff[e_bits]:  # exp_diff is negative, meaning exp_b > exp_a
                final_sign |= sign_b
            with pyrtl.otherwise:  # exp_diff is positive, meaning exp_a > exp_b
                final_sign |= sign_a
                
    return final_sign


In [720]:
def test_detect_final_sign():
    reset_working_block()
    
    # Create inputs
    sign_a = Input(1, 'sign_a')
    sign_b = Input(1, 'sign_b')
    exp_diff = Input(E_BITS + 1, 'exp_diff')
    is_neg = Input(1, 'is_neg')
    
    # Get output
    final_sign = detect_final_sign(sign_a, sign_b, exp_diff, is_neg, E_BITS)
    
    # Create output wire
    sign_out = Output(1, 'sign_out')
    sign_out <<= final_sign
    
    # Simulate
    sim_trace = SimulationTrace()
    sim = Simulation(tracer=sim_trace)
    
    # Test cases covering different scenarios:
    # 1. Same signs (both positive)
    # 2. Same signs (both negative)
    # 3. Different signs, exp_a > exp_b
    # 4. Different signs, exp_b > exp_a
    # 5. Different signs, equal exponents
    inputs = {
        'sign_a':    [0,    1,    0,    1,    0],
        'sign_b':    [0,    1,    1,    0,    1],
        'exp_diff':  [0,    0,    2,    -2,   0],
        'is_neg':    [0,    1,    0,    1,    1]
    }
    
    sim.step_multiple(inputs)
    
    input_repr_map = {
        'sign_a': repr_sign,
        'sign_b': repr_sign,
        'exp_diff': lambda x: str(rev_twos_comp_repr(x, E_BITS + 1)),
        'is_neg': repr_sign,
        'sign_out': repr_sign
    }
    
    trace_list = list(input_repr_map.keys())
    custom_render_trace(sim_trace, trace_list=trace_list, repr_per_name=input_repr_map)

print("Testing sign detection module:")
test_detect_final_sign()


Testing sign detection module:


<IPython.core.display.Javascript object>

### Normalization and Rounding

In [688]:
def normalize_and_round(
    abs_mantissa: WireVector,
    sticky_bit: WireVector,
    guard_bit: WireVector,
    round_bit: WireVector,
    lzc: WireVector,
    m_bits: int
) -> tuple[WireVector, WireVector]:
    """
    Normalize and round the mantissa result
    
    Args:
        abs_mantissa: absolute value of mantissa sum (m_bits + 2 wide)
        sticky_bit: sticky bit from alignment
        guard_bit: guard bit from alignment
        round_bit: round bit from alignment
        lzc: leading zero count from LZD (4 bits wide)
        m_bits: number of mantissa bits in format
        
    Returns:
        norm_mantissa: normalized and rounded mantissa (m_bits wide)
        extra_increment: whether rounding caused an overflow requiring exponent adjustment
    """
    assert len(abs_mantissa) == m_bits + 2
    assert len(sticky_bit) == len(guard_bit) == len(round_bit) == 1
    assert len(lzc) == 4
    
    # Normalize by shifting left according to LZD
    norm_shift = WireVector(m_bits + 2, 'norm_shift')
    norm_shift <<= pyrtl.shift_left_logical(
        abs_mantissa,
        lzc.zero_extended(m_bits + 2)
    )
    
    # Round-to-nearest-even logic
    # Round up if:
    # 1. Guard is 1 AND (Round OR Sticky is 1)
    # 2. Guard is 1 AND Round=Sticky=0 AND LSB=1 (tie case, round to even)
    round_up = WireVector(1, 'round_up')
    lsb = norm_shift[1]  # LSB of normalized mantissa
    
    round_up <<= (
        (guard_bit & (round_bit | sticky_bit)) |
        (guard_bit & ~round_bit & ~sticky_bit & lsb)
    )
    
    # Add rounding increment
    rounded_mantissa = WireVector(m_bits + 2, 'rounded_mantissa')
    rounded_mantissa <<= norm_shift + round_up
    
    # Check if rounding caused overflow
    extra_increment = WireVector(1, 'extra_increment')
    extra_increment <<= rounded_mantissa[m_bits + 1]  # New carry after rounding
    
    # Final mantissa (excluding hidden bit)
    final_mantissa = WireVector(m_bits, 'final_mantissa')
    final_mantissa <<= pyrtl.select(
        extra_increment,
        rounded_mantissa[1:-1],     # Overflow case: lsb+1 -> msb-1 of rounded mantissa TODO: NEEDS ROUNDING!
        rounded_mantissa[:m_bits]   # Normal case: take m_bits of rounded mantissa LSBs
    )
    
    return final_mantissa, extra_increment

In [689]:
def test_normalize_and_round():
    reset_working_block()
    
    # Create inputs
    abs_mantissa = Input(M_BITS + 2, 'abs_mantissa')
    sticky = Input(1, 'sticky')
    guard = Input(1, 'guard')
    round_bit = Input(1, 'round')
    lzc = Input(4, 'lzc')
    
    # Get outputs
    final_mantissa, extra_inc = normalize_and_round(
        abs_mantissa, sticky, guard, round_bit, lzc, M_BITS
    )
    
    # Create output wires
    mant_out = Output(M_BITS, 'mant_out')
    inc_out = Output(1, 'inc_out')
    
    mant_out <<= final_mantissa
    inc_out <<= extra_inc
    
    # Simulate
    sim_trace = SimulationTrace()
    sim = Simulation(tracer=sim_trace)
    
    # Test cases:
    # 1. No leading zeros, round up
    # 2. Two leading zeros, round down
    # 3. One leading zero, tie case
    # 4. Three leading zeros, round up
    # 5. No leading zeros, no rounding needed
    inputs = {
        'abs_mantissa': [0b010000000,  # Normal case
                        0b001100000,  # 2 leading zeros
                        0b001000000,  # 2 leading zeros
                        0b000100000,  # 3 leading zeros
                        0b010000000], # No rounding
        'sticky':       [1, 0, 0, 1, 0],
        'guard':        [1, 0, 1, 1, 0],
        'round':        [0, 0, 0, 0, 0],
        'lzc':          [0, 2, 1, 3, 1]
    }
    
    sim.step_multiple(inputs)
    
    input_repr_map = {
        'abs_mantissa': repr_mantissa_sum,
        'sticky': repr_num,
        'guard': repr_num,
        'round': repr_num,
        'lzc': repr_num,
        'norm_shift': repr_mantissa_sum,
        'rounded_mantissa': repr_mantissa_sum,
        'final_mantissa': repr_mantissa_hidden,
        'inc_out': repr_num
    }
    
    trace_list = list(input_repr_map.keys())
    custom_render_trace(sim_trace, trace_list=trace_list, repr_per_name=input_repr_map)

print("Testing normalization and rounding module:")
test_normalize_and_round()


Testing normalization and rounding module:


<IPython.core.display.Javascript object>

### Exponent Subtractor and Increment

In [699]:
def adjust_final_exponent(
    exp_larger: WireVector,
    lzc: WireVector,
    round_increment: WireVector,
    e_bits: int
) -> WireVector:
    """
    Compute final exponent by subtracting LZC and handling round increment
    
    Args:
        exp_larger: exponent of larger number (e_bits wide)
        lzc: leading zero count from LZD (4 bits wide)
        round_increment: overflow bit from rounding (1 bit)
        e_bits: number of exponent bits in format
        
    Returns:
        final_exp: adjusted exponent value (e_bits wide)
    """
    assert len(exp_larger) == e_bits
    assert len(lzc) == 4
    assert len(round_increment) == 1
    
    # First subtract LZC from larger exponent
    lzc_adjusted = WireVector(e_bits, 'lzc_adjusted')
    lzc_adjusted <<= exp_larger - lzc.zero_extended(e_bits)
    
    # Then add 1 if rounding caused overflow
    final_exp = WireVector(e_bits, 'final_exp')
    final_exp <<= lzc_adjusted + round_increment
    
    return final_exp

In [701]:
def test_adjust_final_exponent():
    reset_working_block()
    
    # Create inputs
    exp_larger = Input(E_BITS, 'exp_larger')
    lzc = Input(4, 'lzc')
    round_inc = Input(1, 'round_inc')
    
    # Get output
    final_exp = adjust_final_exponent(exp_larger, lzc, round_inc, E_BITS)
    
    # Create output wire
    exp_out = Output(E_BITS, 'exp_out')
    exp_out <<= final_exp
    
    # Simulate
    sim_trace = SimulationTrace()
    sim = Simulation(tracer=sim_trace)
    
    # Test cases:
    # 1. Normal case, no round increment
    # 2. Normal case with round increment
    # 3. Large LZC value
    # 4. Near maximum exponent
    # 5. Near minimum exponent
    inputs = {
        'exp_larger': [128,  128,  130,  254,  3],
        'lzc':        [2,    2,    5,    0,    2],
        'round_inc':  [0,    1,    0,    1,    0]
    }
    
    sim.step_multiple(inputs)
    
    input_repr_map = {
        'exp_larger': repr_exp,
        'lzc': repr_num,
        'round_inc': repr_num,
        'lzc_adjusted': repr_exp,
        'exp_out': repr_exp
    }
    
    trace_list = list(input_repr_map.keys())
    custom_render_trace(sim_trace, trace_list=trace_list, repr_per_name=input_repr_map)

print("Testing final exponent adjustment module:")
test_adjust_final_exponent()


Testing final exponent adjustment module:


<IPython.core.display.Javascript object>

### Putting it all together

In [740]:
def stage_5(
    abs_mantissa: WireVector,
    sticky_bit: WireVector,
    guard_bit: WireVector,
    round_bit: WireVector,
    lzc: WireVector,
    exp_larger: WireVector,
    sign_a: WireVector,
    sign_b: WireVector,
    exp_diff: WireVector,
    is_neg: WireVector,
    e_bits: int,
    m_bits: int
) -> tuple[WireVector, WireVector, WireVector]:
    """
    Stage 5: Normalization, Rounding, and Final Adjustments
    
    Args:
        abs_mantissa: absolute value of mantissa sum (m_bits + 2 wide)
        sticky_bit: sticky bit from alignment
        guard_bit: guard bit from alignment
        round_bit: round bit from alignment
        lzc: leading zero count from LZD (4 bits wide)
        exp_larger: larger input exponent (e_bits wide)
        sign_a: sign of first input
        sign_b: sign of second input
        exp_diff: exponent difference (e_bits + 1 wide)
        is_neg: sign from mantissa operation
        e_bits: number of exponent bits in format
        m_bits: number of mantissa bits in format
        
    Returns:
        final_sign: computed sign for result
        final_exp: adjusted exponent value
        final_mantissa: normalized and rounded mantissa
    """
    assert len(abs_mantissa) == m_bits + 2
    assert len(sticky_bit) == len(guard_bit) == len(round_bit) == 1
    assert len(lzc) == 4
    assert len(exp_larger) == e_bits, f"{len(exp_larger)=}, must be {e_bits}"
    assert len(sign_a) == len(sign_b) == len(is_neg) == 1
    assert len(exp_diff) == e_bits + 1
    
    # 1. Normalize and round mantissa
    norm_mantissa, round_inc = normalize_and_round(
        abs_mantissa, sticky_bit, guard_bit, round_bit, lzc, m_bits
    )
    
    # 2. Adjust exponent
    final_exp = adjust_final_exponent(
        exp_larger, lzc, round_inc, e_bits
    )
    
    # 3. Determine final sign
    final_sign = detect_final_sign(
        sign_a, sign_b, exp_diff, is_neg, e_bits
    )
    
    return final_sign, final_exp, norm_mantissa


In [705]:
def test_stage_5():
    reset_working_block()
    
    # Create inputs
    abs_mantissa = Input(M_BITS + 2, 'abs_mantissa')
    sticky = Input(1, 'sticky')
    guard = Input(1, 'guard')
    round_bit = Input(1, 'round')
    lzc = Input(4, 'lzc')
    exp_larger = Input(E_BITS, 'exp_larger')
    sign_a = Input(1, 'sign_a')
    sign_b = Input(1, 'sign_b')
    exp_diff = Input(E_BITS + 1, 'exp_diff')
    is_neg = Input(1, 'is_neg')
    
    # Get outputs
    final_sign, final_exp, final_mant = stage_5(
        abs_mantissa, sticky, guard, round_bit, lzc,
        exp_larger, sign_a, sign_b, exp_diff, is_neg,
        E_BITS, M_BITS
    )
    
    # Create output wires
    sign_out = Output(1, 'sign_out')
    exp_out = Output(E_BITS, 'exp_out')
    mant_out = Output(M_BITS, 'mant_out')
    
    sign_out <<= final_sign
    exp_out <<= final_exp
    mant_out <<= final_mant
    
    # Simulate
    sim_trace = SimulationTrace()
    sim = Simulation(tracer=sim_trace)
    
    # Test cases covering different scenarios:
    # 1. Normal case, no special handling
    # 2. Normalization needed with round up
    # 3. Different signs with exp_a > exp_b
    # 4. Different signs with exp_b > exp_a
    # 5. Round to even case
    inputs = {
        'abs_mantissa': [0b010000000,  # Normal case
                        0b001100000,  # 2 leading zeros
                        0b010000000,  # Different signs
                        0b010000000,  # Different signs
                        0b010000000], # Round to even
        'sticky':       [0, 1, 0, 0, 0],
        'guard':        [0, 1, 0, 0, 1],
        'round':        [0, 0, 0, 0, 0],
        'lzc':          [0, 2, 0, 0, 0],
        'exp_larger':   [128, 128, 130, 126, 128],
        'sign_a':       [0, 0, 0, 1, 0],
        'sign_b':       [0, 0, 1, 0, 0],
        'exp_diff':     [0, 0, 2, -2, 0],
        'is_neg':       [0, 0, 0, 1, 0]
    }
    
    sim.step_multiple(inputs)
    
    input_repr_map = {
        'abs_mantissa': repr_mantissa_sum,
        'sticky': repr_num,
        'guard': repr_num,
        'round': repr_num,
        'lzc': repr_num,
        'exp_larger': repr_exp,
        'sign_a': repr_sign,
        'sign_b': repr_sign,
        'exp_diff': lambda x: str(rev_twos_comp_repr(x, E_BITS + 1)),
        'is_neg': repr_sign,
        'sign_out': repr_sign,
        'exp_out': repr_exp,
        'mant_out': lambda x: format(x, f'0{M_BITS}b')
    }
    
    trace_list = list(input_repr_map.keys())
    custom_render_trace(sim_trace, trace_list=trace_list, repr_per_name=input_repr_map)

print("Testing stage 5:")
test_stage_5()

Testing stage 5:


<IPython.core.display.Javascript object>

## Building the Adder Module

First lets put it all together in one large combinational logic design

In [820]:
def bfloat16_adder(float_a: WireVector, float_b: WireVector, e_bits: int, m_bits: int) -> WireVector:
    sign_a, sign_b, exp_a, exp_b, mantissa_a, mantissa_b = stage_1(float_a, float_b, e_bits, m_bits)
    
    sign_xor, exp_larger, signed_shift, mant_smaller, mant_larger = stage_2(sign_a, sign_b, exp_a, exp_b, mantissa_a, mantissa_b, e_bits, m_bits)

    abs_shift = WireVector(e_bits, 'abs_shift')
    abs_shift <<= signed_shift[:e_bits]

    aligned_mant_msb, sticky_bit, guard_bit, round_bit = stage_3(mant_smaller, abs_shift, m_bits)

    mantissa_sum, is_neg, lzc = stage_4(aligned_mant_msb, mant_larger, sign_xor, m_bits)

    final_sign, final_exp, norm_mantissa = stage_5(mantissa_sum, sticky_bit, guard_bit, round_bit, lzc, exp_larger, sign_a, sign_b, signed_shift, is_neg, e_bits, m_bits)

    float_result = WireVector(e_bits+m_bits+1, 'float_result')
    float_result <<= pyrtl.concat(final_sign, final_exp, norm_mantissa)
    return float_result

In [821]:
def create_bf16_adder_tests(
    n_values: int = 5,
    pipeline_stages: int = 0
) -> tuple[dict[str, list[int]]]:
    
    inputs = torch.rand(2, n_values, dtype=torch.bfloat16)
    outputs = inputs[0] + inputs[1]
    
    if pipeline_stages > 0:
        inputs = torch.concat((inputs, torch.zeros(2, pipeline_stages, dtype=torch.bfloat16)), dim=1)
        outputs = torch.concat((torch.zeros(pipeline_stages, dtype=torch.bfloat16), outputs))

    raw_inputs = inputs.view(torch.uint16)
    raw_outputs = outputs.view(torch.uint16)

    return raw_inputs.tolist(), raw_outputs.tolist()

In [841]:
def test_bfloat16_adder():
    reset_working_block()

    float_a, float_b, gt = Input(16, 'float_a'), Input(16, 'float_b'), Input(16, 'gt')
    float_out = Output(16, 'float_out')

    float_out <<= bfloat16_adder(float_a, float_b, E_BITS, M_BITS)

    sim_trace = SimulationTrace()
    sim = Simulation(tracer=sim_trace)

    ins, outs = create_bf16_adder_tests()
    test_inputs = {
        'float_a': ins[0],
        'float_b': ins[1],
        'gt': outs
    }

    sim.step_multiple(test_inputs)

    trace_list = ['float_a', 'float_b', 'gt', 'float_out']

    custom_render_trace(sim_trace, trace_list=trace_list, repr_func=repr_bf16_tensor)

test_bfloat16_adder()

<IPython.core.display.Javascript object>

In [845]:
pyrtl.synthesize()
pyrtl.area_estimation(tech_in_nm=5)

(2.7579408284023667e-05, 0)

In [847]:
from numpy import sqrt


sqrt(1/2.7579408284023667e-05)

np.float64(190.41779329687313)

In [838]:
basic_circuit_analysis(False, False, 5)

Pre-synthesis Results:
The total block timing delay is  10321.067169990045
Max frequency of block:  3.5931705071292286 MHz
Estimated Area of block 8.696534911242602e-06 mm^2 using 5nm process



In [826]:
with open('bfloat16_adder.svg', 'w+') as f:
    pyrtl.output_to_svg(f)


In [827]:
pyrtl.synthesize()
pyrtl.optimize()
with open('bfloat16_adder_synth.svg', 'w+') as f:
    pyrtl.output_to_svg(f)

Input Wire, gt has been deemed useless by optimization


## Fully Pipelined Design

### PyRTL SimplePipeline Class

In [848]:
class SimplePipeline(object):
    """ Pipeline builder with auto generation of pipeline registers. """

    def __init__(self):
        self._pipeline_register_map = {}
        self._current_stage_num = 0
        stage_list = [method for method in dir(self) if method.startswith('stage')]
        for stage in sorted(stage_list):
            stage_method = getattr(self, stage)
            stage_method()
            self._current_stage_num += 1

    def __getattr__(self, name):
        try:
            return self._pipeline_register_map[self._current_stage_num][name]
        except KeyError:
            raise pyrtl.PyrtlError(
                'error, no pipeline register "%s" defined for stage %d'
                % (name, self._current_stage_num))

    def __setattr__(self, name, value):
        if name.startswith('_'):
            # do not do anything tricky with variables starting with '_'
            object.__setattr__(self, name, value)
        else:
            next_stage = self._current_stage_num + 1
            pipereg_id = str(self._current_stage_num) + 'to' + str(next_stage)
            rname = 'pipereg_' + pipereg_id + '_' + name
            new_pipereg = pyrtl.Register(bitwidth=len(value), name=rname)
            if next_stage not in self._pipeline_register_map:
                self._pipeline_register_map[next_stage] = {}
            self._pipeline_register_map[next_stage][name] = new_pipereg
            new_pipereg.next <<= value

In [857]:
class PipelinedBF16Adder(SimplePipeline):
    def __init__(self):
        # Define inputs and outputs
        self._float_a = pyrtl.Input(E_BITS + M_BITS + 1, 'float_a')
        self._float_b = pyrtl.Input(E_BITS + M_BITS + 1, 'float_b')
        self._result = pyrtl.Output(E_BITS + M_BITS + 1, 'result')
        super(PipelinedBF16Adder, self).__init__()

    def stage0(self):
        """Stage 1: Input Parsing and Extraction"""
        # Extract components from inputs
        self.sign_a, self.sign_b, self.exp_a, self.exp_b, self.mant_a, self.mant_b = \
            stage_1(self._float_a, self._float_b, E_BITS, M_BITS)

    def stage1(self):
        """Stage 2: Exponent Compare and Shift Amount"""
        # Pass through values needed in future stages
        self.sign_a = self.sign_a
        self.sign_b = self.sign_b
        
        # Calculate new values
        self.sign_xor, self.exp_larger, self.signed_shift, self.mant_smaller, self.mant_larger = \
            stage_2(self.sign_a, self.sign_b, self.exp_a, self.exp_b, 
                   self.mant_a, self.mant_b, E_BITS, M_BITS)

    def stage2(self):
        """Stage 3: Mantissa Alignment and SGR Generation"""
        # Pass through values needed in future stages
        self.sign_a = self.sign_a
        self.sign_b = self.sign_b
        self.sign_xor = self.sign_xor
        self.exp_larger = self.exp_larger
        self.signed_shift = self.signed_shift
        self.mant_larger = self.mant_larger

        # Calculate absolute shift amount
        abs_shift = WireVector(E_BITS, 'abs_shift')
        abs_shift <<= self.signed_shift[:E_BITS]
        
        # Perform alignment and generate SGR bits
        self.aligned_mant_msb, self.sticky, self.guard, self.round = \
            stage_3(self.mant_smaller, abs_shift, M_BITS)

    def stage3(self):
        """Stage 4: Mantissa Addition and LZD"""
        # Pass through values needed in future stages
        self.sign_a = self.sign_a
        self.sign_b = self.sign_b
        self.exp_larger = self.exp_larger
        self.signed_shift = self.signed_shift
        self.sticky = self.sticky
        self.guard = self.guard
        self.round = self.round
        
        # Perform mantissa addition and leading zero detection
        self.mant_sum, self.is_neg, self.lzc = \
            stage_4(self.aligned_mant_msb, self.mant_larger, self.sign_xor, M_BITS)

    def stage4(self):
        """Stage 5: Normalization, Rounding, and Final Assembly"""
        # Calculate final values
        final_sign, final_exp, norm_mantissa = \
            stage_5(self.mant_sum, self.sticky, self.guard, self.round,
                   self.lzc, self.exp_larger, self.sign_a, self.sign_b,
                   self.signed_shift, self.is_neg, E_BITS, M_BITS)
        
        # Concatenate final result
        self._result <<= pyrtl.concat(final_sign, final_exp, norm_mantissa)

def test_pipelined_adder():
    reset_working_block()
    
    # Instantiate the pipeline
    adder = PipelinedBF16Adder()
    
    # Create simulation
    sim_trace = SimulationTrace()
    sim = Simulation(tracer=sim_trace)
    
    # Test vectors
    input_a = [0b0100000001000000, 0b0100000010000000, 0b0100000001000000]
    input_b = [0b0100000010000000, 0b0100000001000000, 0b1100000001000000]
    
    # Create input dictionary for multiple steps
    inputs = {
        'float_a': input_a,
        'float_b': input_b
    }
    
    # Run simulation
    sim.step_multiple(inputs)
    
    # Add extra steps to flush pipeline
    sim.step_multiple({'float_a': [0] * 5, 'float_b': [0] * 5})
    
    # Display results
    input_repr_map = {
        'float_a': repr_bf16_tensor,
        'float_b': repr_bf16_tensor,
        'result': repr_bf16_tensor
    }
    
    trace_list = ['float_a', 'float_b', 'result']
    custom_render_trace(sim_trace, trace_list=trace_list, repr_per_name=input_repr_map)

print("Testing pipelined BF16 adder:")
test_pipelined_adder()

basic_circuit_analysis(tech_in_nm=32)

Testing pipelined BF16 adder:


<IPython.core.display.Javascript object>

Pre-synthesis Results:
The total block timing delay is  3892.8009697246603
Max frequency of block:  57.56906083720198 MHz
Estimated Area of block 0.0006317608354082841 mm^2 using 32nm process

Synthesis Results:
The total block timing delay is  7778.570000000006
Max frequency of block:  30.160109654618655 MHz
Estimated Area of block 0.0014067869538461537 mm^2 using 32nm process

The total block timing delay is  5052.610000000001
Max frequency of block:  45.285413440965435 MHz
Estimated Area of block 0.000746732016852071 mm^2 using 32nm process

